In [0]:
%run ./99_utils

✅ Clarity AML utilities loaded


✅ Snowflake connection successful!
   User:      VEDOXO123
   Database:  CLARITY_AML
   Warehouse: CLARITY_WH


In [0]:
setup_adls()
spark.conf.set("spark.databricks.delta.preview.enabled", "true")
print("✅ Setup complete")

✅ ADLS Gen2 connected: clarityadls
✅ Setup complete


In [0]:
%run ./01_fuzzy_entity_matching

✅ Clarity AML utilities loaded


✅ Snowflake connection successful!
   User:      VEDOXO123
   Database:  CLARITY_AML
   Warehouse: CLARITY_WH


✅ ADLS Gen2 connected: clarityadls


✅ Clarity AML utilities loaded


Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
  Using cached thefuzz-0.22.1-py3-none-any.whl (8.2 kB)
  Using cached python_levenshtein-0.27.3-py3-none-any.whl (9.5 kB)
  Using cached rapidfuzz-3.14.5-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (3.2 MB)
  Using cached levenshtein-0.27.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (153 kB)
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


...

📂 Loading KvK company registry...
✅ Loaded: 50,000 KvK companies
+----------+-------------------------------------------+----------+-----------+----------------------+---------+--------+--------------+------------+-----------------+---------+------------------+--------------+-------------------------------------------+
|kvk_number|company_name                               |legal_form|sector_code|sector_description    |city     |postcode|street        |house_number|registration_date|status   |annual_revenue_eur|employee_count|company_name_clean                         |
+----------+-------------------------------------------+----------+-----------+----------------------+---------+--------+--------------+------------+-----------------+---------+------------------+--------------+-------------------------------------------+
|53269655  |Bosch Restaurants BV                       |BV        |5610       |Restaurants           |Almere   |8120 ZA |Jasonhof      |325         |2023-12-25       |

✅ Cash intensity anomalies detected: 0
+-----------+----------+-----------------------+--------------+
|sender_name|amount_eur|max_plausible_daily_eur|employee_count|
+-----------+----------+-----------------------+--------------+
+-----------+----------+-----------------------+--------------+



✅ Fuzzy matching functions defined

Test examples:
  'BANCO NACIONAL CUBA' vs 'BANCO NACIONAL DE CUBA': 0.91
  'TIMCHENKO TRADING BV' vs 'TIMCHENKO': 0.77
  'VAN DER BERG BV' vs 'APPLE INC': 0.29


✅ Broadcasted 19,043 sanctions names to all workers


✅ UDFs registered
   Match threshold: 0.85


In [0]:
# GraphFrames installed as cluster library
# No pip install needed
print("✅ GraphFrames available as cluster library")

✅ GraphFrames available as cluster library


🔍 Step 1: Exact matching...
   Exact sanctions hits: 125
+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+------------+----------+--------+------------+----------------+----------+------------+-------------+-------------+------------+--------------------+--------------------+----------------+------------------+------------------+-------------------+--------------------+--------------------+----------+-------------+---------------+----------------+--------------+---------------------+----------------------+----------------+-----------+---------------------+-----------------------+--------------+----------------------+-----------------+-----------------+-------------------+--------------------+----------+------+--------------------+-----------------+---------+--------------+----------+
|   sender_name_clean|         sender_iban|       receiver_iban|      transaction_id|         sender_name|sender_bic


🔍 Step 2: Optimised fuzzy matching...
   Candidate pairs: 2,978
   Fuzzy matches found: 0
✅ Fuzzy sanctions hits: 0


🚨 Sanctions hits found:
+--------------+-----------+-----------------+------------------+---------------------+----------+
|transaction_id|sender_name|sender_name_clean|fuzzy_matched_name|fuzzy_sanctions_score|amount_eur|
+--------------+-----------+-----------------+------------------+---------------------+----------+
+--------------+-----------+-----------------+------------------+---------------------+----------+




🔗 Combining exact and fuzzy results...
✅ Risk scores computed
+---------+-----+
|risk_tier|count|
+---------+-----+
|     HIGH|   67|
|      LOW| 2302|
|   MEDIUM|  319|
+---------+-----+

✅ Written to Silver enriched: abfss://silver@clarityadls.dfs.core.windows.net/transactions_enriched
   Total records written: 2,688


In [0]:
from datetime import datetime, timedelta
from pyspark.sql.functions import col

process_date = datetime.now()

YEAR  = process_date.year
MONTH = process_date.month
DAY   = process_date.day

print(f"📅 Processing date: {YEAR}-{MONTH:02d}-{DAY:02d}")

# ── Read from transactions_enriched (NB1 output) ───────────────
# This has sanctions_hit, kvk_registered flags we need

BACKFILL_MODE = True   # set to False after first run

if BACKFILL_MODE:
        print("⚠️  BACKFILL MODE — loading ALL partitions from Silver")
        transactions_raw = spark.read.parquet(silver("transactions_enriched"))
        raw_count = transactions_raw.count()

        # ── Deduplicate on transaction_id ──────────────────────────────
        # Silver layer is append-only — if NB1 ran multiple times today
        # or Kafka batch was reprocessed, same transaction_id appears
        # in multiple Parquet files. Keep only one record per ID.
        # transaction_id is a UUID generated at Kafka produce time —
        # it is the guaranteed unique identifier per transaction.

        transactions = transactions_raw.dropDuplicates(["transaction_id"])

        record_count = transactions.count()
        duplicates_removed = raw_count - record_count

        print(f"✅ Loaded:              {raw_count:,} raw records")
        print(f"✅ After dedup:         {record_count:,} unique transactions")
        if duplicates_removed > 0:
            print(f"⚠️  Duplicates removed: {duplicates_removed:,} "
                f"(NB1 ran {raw_count // record_count}x today)")
        else:
            print(f"✅ No duplicates found")

        if record_count == 0:
            print("⚠️  No records for this date in transactions_enriched.")
            print("   Make sure NB1 fuzzy matching ran first.")
            dbutils.notebook.exit("NO_DATA")
else:
        transactions_raw = spark.read.parquet(
            silver("transactions_enriched")
        ).filter(
            (col("value_date_year")  == YEAR)  &
            (col("value_date_month") == MONTH) &
            (col("value_date_day")   == DAY)
        )

        raw_count = transactions_raw.count()

        # ── Deduplicate on transaction_id ──────────────────────────────
        # Silver layer is append-only — if NB1 ran multiple times today
        # or Kafka batch was reprocessed, same transaction_id appears
        # in multiple Parquet files. Keep only one record per ID.
        # transaction_id is a UUID generated at Kafka produce time —
        # it is the guaranteed unique identifier per transaction.

        transactions = transactions_raw.dropDuplicates(["transaction_id"])

        record_count = transactions.count()
        duplicates_removed = raw_count - record_count

        print(f"✅ Loaded:              {raw_count:,} raw records")
        print(f"✅ After dedup:         {record_count:,} unique transactions")
        if duplicates_removed > 0:
            print(f"⚠️  Duplicates removed: {duplicates_removed:,} "
                f"(NB1 ran {raw_count // record_count}x today)")
        else:
            print(f"✅ No duplicates found")

        if record_count == 0:
            print("⚠️  No records for this date in transactions_enriched.")
            print("   Make sure NB1 fuzzy matching ran first.")
            dbutils.notebook.exit("NO_DATA")

📅 Processing date: 2026-07-03
⚠️  BACKFILL MODE — loading ALL partitions from Silver
✅ Loaded:              2,688 raw records
✅ After dedup:         1,569 unique transactions
⚠️  Duplicates removed: 1,119 (NB1 ran 1x today)


In [0]:
import pyspark.sql.functions as F

# ── Per-account statistical baseline from full Silver history ──
# Detects clean-trail laundering: accounts whose recent behavior
# is statistically inconsistent with their own past.
# Z-score > 3 means this amount is in top 0.1% of their own history.

print("📊 Computing per-account statistical baselines...")

df_full_history = spark.read.parquet(silver("transactions_enriched")) \
    .dropDuplicates(["transaction_id"]) \
    .select("sender_iban", "amount_eur", "ingestion_ts")

# Per-account stats across ALL history
df_account_stats = df_full_history.groupBy("sender_iban").agg(
    F.avg("amount_eur").alias("hist_mean_amount"),
    F.stddev("amount_eur").alias("hist_stddev_amount"),
    F.count("*").alias("hist_txn_count"),
    F.max("ingestion_ts").alias("last_seen_ts"),
    F.min("ingestion_ts").alias("first_seen_ts")
).withColumn(
    # Account age in days — young accounts doing large transactions
    # have no established pattern to hide behind
    "account_age_days",
    (F.col("last_seen_ts") - F.col("first_seen_ts")) / (1000 * 86400)
).withColumn(
    # Protect against null/zero stddev (single-transaction accounts)
    "hist_stddev_amount",
    F.when(
        F.col("hist_stddev_amount").isNull() |
        (F.col("hist_stddev_amount") == 0),
        F.col("hist_mean_amount") * 0.1
    ).otherwise(F.col("hist_stddev_amount"))
)

# Join stats onto today's transactions
transactions = transactions.join(
    df_account_stats,
    transactions["sender_iban"] == df_account_stats["sender_iban"],
    "left"
).drop(df_account_stats["sender_iban"]) \
.withColumn(
    # How many std deviations above this account's own mean is this amount?
    "amount_zscore",
    F.when(
        F.col("hist_stddev_amount").isNotNull() &
        (F.col("hist_stddev_amount") > 0),
        (F.col("amount_eur") - F.col("hist_mean_amount")) /
        F.col("hist_stddev_amount")
    ).otherwise(0.0)
).withColumn(
    # z > 3 = statistically anomalous for THIS account's own history
    "statistical_anomaly",
    F.col("amount_zscore") > 3.0
).withColumn(
    # Brand new account (<30 days old) doing high-value transactions
    "young_account_high_value",
    (F.col("account_age_days") < 30) &
    (F.col("amount_eur") > 10000)
)

stat_count  = transactions.filter(F.col("statistical_anomaly")).count()
young_count = transactions.filter(F.col("young_account_high_value")).count()
print(f"✅ Statistical anomalies (z > 3):     {stat_count:,}")
print(f"✅ Young account high-value flags:     {young_count:,}")

📊 Computing per-account statistical baselines...
✅ Statistical anomalies (z > 3):     3
✅ Young account high-value flags:     396


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import lit, broadcast
print("🔨 Building transaction graph...")


# ── NODES ─────────────────────────────────────────────────────
# Every unique IBAN is a node in the graph
# We combine sender and receiver IBANs into one node list

# ── NODES ─────────────────────────────────────────────────────

sender_nodes = transactions.select(
    col("sender_iban").alias("id"),
    col("sender_name").alias("name"),
    col("sender_name_clean").alias("name_clean"),
    col("sender_bic").alias("bic"),
    col("sanctions_hit"),
    col("kvk_registered"),
    col("kvk_status")
)

# ── FIX: screen receivers against sanctions list too ──────────
# The original code hardcoded lit(False) for receiver sanctions_hit.
# A sanctioned entity that ONLY receives funds was never flagged.
# We do a left join on receiver_name_clean just like NB01 does for senders.

receiver_sanctions = transactions.select(
    col("receiver_iban").alias("id"),
    col("receiver_name").alias("name"),
    col("receiver_name_clean").alias("name_clean"),
    col("receiver_bic").alias("bic")
).join(
    broadcast(sanctions.select(
        col("entity_name_clean"),
        col("sanctions_program")
    )),
    col("name_clean") == col("entity_name_clean"),
    "left"
).withColumn(
    "sanctions_hit",
    col("sanctions_program").isNotNull()
).withColumn(
    "kvk_registered", lit(False)
).withColumn(
    "kvk_status", lit("UNKNOWN")
).drop("entity_name_clean", "sanctions_program")

receiver_nodes = receiver_sanctions

# Union sender and receiver nodes, keep one record per IBAN
# F.max("sanctions_hit") means if EITHER the sender OR receiver
# join flagged this IBAN, it stays flagged in the final node table.
nodes = sender_nodes.unionByName(receiver_nodes) \
    .groupBy("id") \
    .agg(
        F.first("name").alias("name"),
        F.first("name_clean").alias("name_clean"),
        F.first("bic").alias("bic"),
        F.max("sanctions_hit").alias("sanctions_hit"),     # ← max = True wins
        F.max("kvk_registered").alias("kvk_registered"),
        F.first("kvk_status").alias("kvk_status")
    )

node_count = nodes.count()
print(f"✅ Nodes (unique IBANs): {node_count:,}")

# Sanity check — how many receiver-only sanctioned nodes did we catch?
receiver_sanction_hits = nodes.filter(
    col("sanctions_hit") == True
).count()
print(f"🚨 Total sanctioned nodes (senders + receivers): {receiver_sanction_hits:,}")

🔨 Building transaction graph...
✅ Nodes (unique IBANs): 2,541
🚨 Total sanctioned nodes (senders + receivers): 124


In [0]:
# ── EDGES ─────────────────────────────────────────────────────
# Every transaction is an edge between sender and receiver
# Used for: structuring, layering, scoring (today's activity only)
# (The 30-day rolling edge set below is built separately, only for circular detection)

print("🔨 Building today's transaction edges...")

edges = transactions.select(
    col("sender_iban").alias("src"),
    col("receiver_iban").alias("dst"),
    col("transaction_id"),
    col("amount_eur"),
    col("purpose_code"),
    col("value_date"),
    col("sanctions_hit"),
    col("statistical_anomaly"),
    col("young_account_high_value")
)

edge_count = edges.count()
print(f"✅ Edges (today's transactions): {edge_count:,}")

🔨 Building today's transaction edges...
✅ Edges (today's transactions): 1,569


In [0]:
# ── Rolling window edges for circular detection ────────────────
# Circular transactions span multiple days.
# A → B on day 1, B → C on day 2, C → A on day 3.
# Today's edges alone never show the full cycle.
# We build a separate 30-day edge set used ONLY for
# circular detection — not for any other metrics.
# All other detection (structuring, layering, scoring)
# still uses today's edges only for accuracy.

CIRCULAR_LOOKBACK_DAYS = 30

print(f"\n📅 Building {CIRCULAR_LOOKBACK_DAYS}-day rolling edges "
      f"for circular detection...")

edges_rolling = spark.read.parquet(
    silver("transactions_enriched")
).dropDuplicates(["transaction_id"]) \
.filter(
    # Keep last 30 days including today
    F.to_date(F.to_timestamp(col("ingestion_ts") / 1000)) >=
    F.date_sub(F.current_date(), CIRCULAR_LOOKBACK_DAYS)
).select(
    col("sender_iban").alias("src"),
    col("receiver_iban").alias("dst"),
    col("transaction_id"),
    col("amount_eur"),
    col("purpose_code"),
    col("value_date")
)

rolling_edge_count = edges_rolling.count()
print(f"✅ Rolling edges (30 days): {rolling_edge_count:,}")
print(f"   Today's edges:           {edge_count:,}")
print(f"   Prior days included:     {rolling_edge_count - edge_count:,}")


📅 Building 30-day rolling edges for circular detection...
✅ Rolling edges (30 days): 1,526
   Today's edges:           1,569
   Prior days included:     -43


In [0]:
print("🔍 Finding connected clusters (undirected graph)...")

# ── Convert directed edges to undirected ───────────────────────
# For each A→B edge, also create B→A edge
# This makes the graph undirected — connection works both ways

directed_edges = edges.select(
    col("src").alias("id"),
    col("dst").alias("neighbor")
)

# Reverse edges — B→A for every A→B
reversed_edges = edges.select(
    col("dst").alias("id"),
    col("src").alias("neighbor")
)

# Combine both directions
undirected_edges = directed_edges.unionByName(
    reversed_edges
).distinct()  # Remove duplicates

print(f"✅ Directed edges:   {directed_edges.count():,}")
print(f"✅ Undirected edges: {undirected_edges.count():,} (both directions)")

# ── Custom connected components ────────────────────────────────
components = nodes.select(
    col("id"),
    col("id").alias("component")
)

for i in range(5):
    updated = components.alias("c").join(
        undirected_edges.alias("e"),   # ← undirected now
        col("c.id") == col("e.id"),
        "left"
    ).join(
        components.select(
            col("id").alias("neighbor"),
            col("component").alias("neighbor_component")
        ),
        "neighbor",
        "left"
    ).withColumn(
        "new_component",
        F.when(
            col("neighbor_component").isNull(),
            col("c.component")
        ).otherwise(
            F.least(col("c.component"), col("neighbor_component"))
        )
    ).groupBy("c.id").agg(
        F.min("new_component").alias("component")
    )

    changes = updated.join(
        components.select(
            col("id"),
            col("component").alias("old_component")
        ),
        "id"
    ).filter(
        col("component") != col("old_component")
    ).count()

    components = updated
    print(f"   Iteration {i+1}: {changes} nodes updated")

    if changes == 0:
        print(f"   Converged after {i+1} iterations ✅")
        break

cluster_count = components.select("component").distinct().count()
print(f"\n✅ Total clusters found: {cluster_count:,}")

print("\nCluster size distribution:")
components.groupBy("component") \
    .count() \
    .groupBy("count") \
    .agg(F.count("*").alias("num_clusters")) \
    .orderBy(col("count").desc()) \
    .show(10)

🔍 Finding connected clusters (undirected graph)...
✅ Directed edges:   1,569
✅ Undirected edges: 2,774 (both directions)
   Iteration 1: 1299 nodes updated
   Iteration 2: 221 nodes updated
   Iteration 3: 4 nodes updated
   Iteration 4: 1 nodes updated
   Iteration 5: 0 nodes updated
   Converged after 5 iterations ✅

✅ Total clusters found: 1,156

Cluster size distribution:
+-----+------------+
|count|num_clusters|
+-----+------------+
|   47|           1|
|   40|           1|
|   34|           1|
|   33|           1|
|   15|           2|
|   14|           1|
|   13|           2|
|    9|           1|
|    8|           2|
|    6|           1|
+-----+------------+
only showing top 10 rows



In [0]:
print("\n1️⃣  Detecting structuring + fan-in accumulation...")

# ── Classic structuring: near-threshold amounts ────────────────
near_threshold = edges.filter(
    (col("amount_eur") >= 7000) & (col("amount_eur") < 10000)
)

structuring_by_sender = near_threshold.groupBy("src").agg(
    F.count("transaction_id").alias("near_threshold_txn_count"),
    F.sum("amount_eur").alias("near_threshold_total"),
    F.countDistinct("dst").alias("unique_receivers")
).filter(
    col("near_threshold_txn_count") >= 3
).withColumn("structuring_role", lit("SENDER")) \
 .withColumn("structuring_account", col("src")) \
 .drop("src")

structuring_by_receiver = near_threshold.groupBy("dst").agg(
    F.count("transaction_id").alias("near_threshold_txn_count"),
    F.sum("amount_eur").alias("near_threshold_total"),
    F.countDistinct("src").alias("unique_senders")
).filter(
    col("near_threshold_txn_count") >= 3
).withColumn("structuring_role", lit("RECEIVER")) \
 .withColumn("structuring_account", col("dst")) \
 .drop("dst")

# ── Fan-in accumulation from full Silver history ───────────────
# Catches: many people sending small amounts to one account
# over weeks/months. Each transaction innocent. Pattern criminal.
# Four signals required simultaneously to avoid flagging
# legitimate landlords, freelance platforms, family pools.

df_all_txn_history = spark.read.parquet(
    silver("transactions_enriched")
).dropDuplicates(["transaction_id"]) \
 .select(
    "sender_iban", "receiver_iban",
    "amount_eur", "ingestion_ts", "purpose_code"
)

# Inbound profile per receiver
inbound_profile = df_all_txn_history.groupBy("receiver_iban").agg(
    F.countDistinct("sender_iban").alias("unique_senders_total"),
    F.sum("amount_eur").alias("total_received_eur"),
    F.count("*").alias("total_inbound_txn_count"),
    F.avg("amount_eur").alias("avg_inbound_amount"),
    F.stddev("ingestion_ts").alias("inbound_timing_stddev"),
    F.countDistinct("purpose_code").alias("inbound_purpose_variety")
)

# Outbound profile — does money leave quickly?
outbound_profile = df_all_txn_history.groupBy("sender_iban").agg(
    F.countDistinct("receiver_iban").alias("unique_receivers_out"),
    F.sum("amount_eur").alias("total_sent_eur"),
    F.count("*").alias("total_outbound_txn_count")
)

# Sender relationship depth — are senders repeat or one-time?
sender_history = df_all_txn_history.groupBy(
    "sender_iban", "receiver_iban"
).agg(F.count("*").alias("pair_txn_count"))

sender_relationship_age = sender_history.groupBy("receiver_iban").agg(
    F.avg("pair_txn_count").alias("avg_sender_relationship_depth"),
    F.avg(
        F.when(col("pair_txn_count") == 1, 1.0).otherwise(0.0)
    ).alias("first_time_sender_ratio")
)

# Combine and score with 4 signals
fan_in_scored = inbound_profile.join(
    outbound_profile,
    inbound_profile["receiver_iban"] == outbound_profile["sender_iban"],
    "left"
).join(
    sender_relationship_age, "receiver_iban", "left"
).withColumn(
    # Signal 1: money leaves fast — kept less than 20%
    "s1_low_retention",
    F.when(
        col("total_sent_eur").isNotNull() &
        (col("total_received_eur") > 0),
        (F.lit(1.0) -
         (col("total_sent_eur") / col("total_received_eur"))) < 0.2
    ).otherwise(False)
).withColumn(
    # Signal 2: outbound goes to very few accounts (1-2)
    # landlord pays 10+ vendors, criminal pays 1-2
    "s2_concentrated_outbound",
    col("unique_receivers_out").isNotNull() &
    (col("unique_receivers_out") <= 2) &
    (col("total_sent_eur") > 10000)
).withColumn(
    # Signal 3: most senders are first-time relationships
    # tenants appear monthly, criminal mules appear once
    "s3_mostly_first_time_senders",
    F.coalesce(col("first_time_sender_ratio"), F.lit(0.0)) > 0.7
).withColumn(
    # Signal 4: no rent or salary purpose codes
    # legitimate multi-sender receivers use RENT, SALA
    # criminal receives CASH, TRAD, INTC
    "s4_no_rent_or_salary_purpose",
    col("inbound_purpose_variety") <= 2
).withColumn(
    "fan_in_signal_count",
    (F.when(col("s1_low_retention"),             1).otherwise(0)) +
    (F.when(col("s2_concentrated_outbound"),     1).otherwise(0)) +
    (F.when(col("s3_mostly_first_time_senders"), 1).otherwise(0)) +
    (F.when(col("s4_no_rent_or_salary_purpose"), 1).otherwise(0))
).withColumn(
    "is_fan_in_suspicious",
    # Base: many senders, large cumulative, small per-transaction
    (col("unique_senders_total") >= 5) &
    (col("total_received_eur") >= 50000) &
    (col("avg_inbound_amount") < 5000) &
    # Plus at least 3 of 4 contextual signals
    (col("fan_in_signal_count") >= 3)
)

fan_in_accumulation = fan_in_scored.filter(
    col("is_fan_in_suspicious") == True
).withColumn("structuring_role", lit("FAN_IN_ACCUMULATION")) \
 .withColumn("structuring_account", col("receiver_iban")) \
 .withColumn("near_threshold_txn_count", col("total_inbound_txn_count")) \
 .withColumn("near_threshold_total", col("total_received_eur")) \
 .select(
    "structuring_account",
    "near_threshold_txn_count",
    "near_threshold_total",
    "structuring_role"
)

# ── Combine all three ──────────────────────────────────────────
structuring_flagged = structuring_by_sender.select(
    "structuring_account", "near_threshold_txn_count",
    "near_threshold_total", "structuring_role"
).unionByName(
    structuring_by_receiver.select(
        "structuring_account", "near_threshold_txn_count",
        "near_threshold_total", "structuring_role"
    )
).unionByName(fan_in_accumulation).distinct()

print(f"   ✅ Classic structuring (sender): {structuring_by_sender.count():,}")
print(f"   ✅ Classic structuring (receiver): {structuring_by_receiver.count():,}")
print(f"   ✅ Fan-in accumulation: {fan_in_accumulation.count():,}")
print(f"   ✅ Total structuring flagged: {structuring_flagged.count():,}")
structuring_flagged.show(10, truncate=False)


# ══════════════════════════════════════════════════════════════
# PATTERN 2 — LAYERING DETECTION
# Signature: chain-shaped nodes (degree=2) within a cluster
# One connection in, one connection out
# ══════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════
# PATTERN 2 — LAYERING DETECTION (rolling 14-day window)
# Layering spans multiple days: A→B on day 1, B→C day 3, C→D day 5
# Today's edges alone miss multi-day chains entirely.
# We use a 14-day rolling edge set for chain detection,
# then confirm with purpose code coherence check.
# ══════════════════════════════════════════════════════════════

print("\n2️⃣  Detecting layering (rolling 14-day chain + purpose coherence)...")

LAYERING_LOOKBACK_DAYS = 14

# ── Build 14-day rolling edges for layering ────────────────────
# Separate from the 30-day circular window — layering chains
# typically form within 2 weeks. Using 30 days adds noise.

edges_rolling_layering = spark.read.parquet(
    silver("transactions_enriched")
).dropDuplicates(["transaction_id"]) \
.filter(
    F.to_date(F.to_timestamp(col("ingestion_ts") / 1000)) >=
    F.date_sub(F.current_date(), LAYERING_LOOKBACK_DAYS)
).select(
    col("sender_iban").alias("src"),
    col("receiver_iban").alias("dst"),
    col("transaction_id"),
    col("purpose_code"),
    col("amount_eur"),
    col("value_date")
)

rolling_layering_edge_count = edges_rolling_layering.count()
print(f"   Rolling edges (14 days): {rolling_layering_edge_count:,}")
print(f"   Today's edges:           {edge_count:,}")
print(f"   Prior days included:     {rolling_layering_edge_count - edge_count:,}")

# ── Build undirected rolling edges ─────────────────────────────
# Degree is undirected — one in, one out — regardless of direction

rolling_directed_l = edges_rolling_layering.select(
    col("src").alias("id"),
    col("dst").alias("neighbor")
)
rolling_reversed_l = edges_rolling_layering.select(
    col("dst").alias("id"),
    col("src").alias("neighbor")
)
rolling_undirected_layering = rolling_directed_l.unionByName(
    rolling_reversed_l
).distinct()

# ── Build rolling components for layering graph ────────────────
# We need a separate component assignment for the 14-day graph
# because today's components don't include multi-day connections.

rolling_nodes_l = rolling_undirected_layering.select("id").distinct()

rolling_components_l = rolling_nodes_l.withColumn(
    "component", col("id")
)

for i in range(8):  # More iterations for larger rolling graph
    updated_l = rolling_components_l.alias("c").join(
        rolling_undirected_layering.alias("e"),
        col("c.id") == col("e.id"),
        "left"
    ).join(
        rolling_components_l.select(
            col("id").alias("neighbor"),
            col("component").alias("neighbor_component")
        ),
        "neighbor",
        "left"
    ).withColumn(
        "new_component",
        F.when(
            col("neighbor_component").isNull(),
            col("c.component")
        ).otherwise(
            F.least(col("c.component"), col("neighbor_component"))
        )
    ).groupBy("c.id").agg(
        F.min("new_component").alias("component")
    )

    changes_l = updated_l.join(
        rolling_components_l.select(
            col("id"),
            col("component").alias("old_component")
        ),
        "id"
    ).filter(col("component") != col("old_component")).count()

    rolling_components_l = updated_l

    if changes_l == 0:
        print(f"   Rolling layering components converged after {i+1} iterations ✅")
        break

# ── Degree on rolling graph ────────────────────────────────────
# A degree-2 node in the rolling graph = pass-through account
# across multiple days. This is the core layering signature.

chain_metrics_rolling = rolling_undirected_layering.groupBy("id").agg(
    F.countDistinct("neighbor").alias("degree")
)

# degree == 2: exactly one in, one out (pass-through)
# We also catch degree-1 terminal nodes that are endpoints
# of confirmed chains — they're not pass-throughs themselves
# but they anchor a chain we've already detected.
chain_nodes_rolling = chain_metrics_rolling.filter(col("degree") == 2)
chain_node_count_rolling = chain_nodes_rolling.count()

print(f"   ✅ Rolling chain-like nodes (degree=2): {chain_node_count_rolling:,}")

# ── Group chain nodes into clusters using rolling components ───
layering_by_cluster = chain_nodes_rolling.join(
    rolling_components_l, "id"
).groupBy("component").agg(
    F.count("id").alias("layering_chain_length")
).filter(col("layering_chain_length") >= 3)
# Minimum 3 = A→B→C→D, which is 2 pass-throughs + endpoints
# A 2-node chain is just a bilateral payment — not layering

# ── Purpose code coherence check ──────────────────────────────
# This separates real supply chains from criminal layering.
#
# Legitimate supply chain node (wholesaler → distributor → retailer):
#   SUPP, GDDS, TRAD — real trade purpose codes
#   Different amounts at each hop (markup exists)
#   Established counterparty relationships
#
# Criminal layering node:
#   INTC (intra-company between unrelated entities) or CASH
#   Same amount minus a small fee at each hop (no value-add)
#   First-time counterparties at every hop
#
# We use the rolling edges for purpose check too —
# a node that only used INTC over 14 days is more damning
# than one INTC transaction today.

chain_purpose = edges_rolling_layering.join(
    chain_nodes_rolling.select(col("id").alias("src")), "src"
).groupBy("src").agg(
    F.countDistinct("purpose_code").alias("purpose_variety"),
    F.collect_set("purpose_code").alias("purpose_codes"),
    F.count("*").alias("chain_txn_count"),
    F.stddev("amount_eur").alias("amount_stddev"),
    F.avg("amount_eur").alias("amount_avg"),
    F.countDistinct("dst").alias("unique_counterparties"),
    # First-time counterparty ratio — criminal nodes meet new accounts every hop
    F.count("*").alias("total_txns_l")
).withColumn(
    # INTC or CASH only = hiding relationship or moving cash
    "only_intc_or_cash",
    (col("purpose_variety") == 1) & (
        F.array_contains(col("purpose_codes"), F.lit("INTC")) |
        F.array_contains(col("purpose_codes"), F.lit("CASH"))
    )
).withColumn(
    # CV < 5% = mechanically consistent amounts = pass-through fee structure
    # Real distributors have price variation. Criminals charge fixed fees.
    "suspicious_amount_consistency",
    F.when(
        col("amount_avg") > 0,
        (F.coalesce(col("amount_stddev"), F.lit(0.0)) /
         col("amount_avg")) < 0.05
    ).otherwise(False)
).withColumn(
    # Counterparties all different = new mule at every hop
    # Legitimate wholesaler reuses same suppliers month after month
    "all_unique_counterparties",
    col("unique_counterparties") == col("chain_txn_count")
).withColumn(
    # Node is suspicious if ANY two of three signals fire together
    # Requiring all three would miss sophisticated layering
    "suspicious_signal_count",
    (F.when(col("only_intc_or_cash"),            1).otherwise(0)) +
    (F.when(col("suspicious_amount_consistency"), 1).otherwise(0)) +
    (F.when(col("all_unique_counterparties"),     1).otherwise(0))
).withColumn(
    "chain_node_suspicious",
    col("suspicious_signal_count") >= 2
)

# ── Count suspicious chain nodes per cluster ───────────────────
suspicious_chain_per_cluster = chain_purpose.filter(
    col("chain_node_suspicious") == True
).join(
    rolling_components_l.select(col("id").alias("src"), "component"),
    "src"
).groupBy("component").agg(
    F.count("src").alias("suspicious_purpose_chain_nodes")
)

# ── Enrich layering clusters with purpose coherence ───────────
layering_by_cluster = layering_by_cluster.join(
    suspicious_chain_per_cluster,
    "component", "left"
).fillna(0, subset=["suspicious_purpose_chain_nodes"]).withColumn(
    # Confirmed layering requires:
    # 3+ chain nodes AND 70%+ of them using suspicious purposes
    # 50% threshold is too loose — a real supply chain with one
    # suspicious node in three would get flagged.
    # 70% means the criminal pattern is dominant in this cluster.
    "layering_confirmed",
    (col("layering_chain_length") >= 3) &
    (col("suspicious_purpose_chain_nodes") >=
     (col("layering_chain_length") * 0.7))
).withColumn(
    # Flag whether this is multi-day — key for bank reporting
    # If rolling edge count >> today's edge count, chain spans days
    "spans_multiple_days",
    F.lit(rolling_layering_edge_count > edge_count)
)

confirmed_layering_count = layering_by_cluster.filter(
    col("layering_confirmed") == True
).count()
total_layering_clusters  = layering_by_cluster.count()

print(f"   ✅ Clusters with chain pattern:          {total_layering_clusters:,}")
print(f"   ✅ Confirmed layering (≥70% suspicious): {confirmed_layering_count:,}")
print(f"   ℹ️  Unconfirmed (likely supply chains):  "
      f"{total_layering_clusters - confirmed_layering_count:,}")
print(f"   ℹ️  Multi-day chains detected:           "
      f"{'YES' if rolling_layering_edge_count > edge_count else 'NO'}")

# ── For cluster scoring, only use confirmed layering ──────────
# Replace the raw layering_by_cluster downstream with this
layering_by_cluster_confirmed = layering_by_cluster.filter(
    col("layering_confirmed") == True
)

layering_by_cluster.show(10)


1️⃣  Detecting structuring + fan-in accumulation...
   ✅ Classic structuring (sender): 1
   ✅ Classic structuring (receiver): 6
   ✅ Fan-in accumulation: 0
   ✅ Total structuring flagged: 7
+-------------------+------------------------+--------------------+----------------+
|structuring_account|near_threshold_txn_count|near_threshold_total|structuring_role|
+-------------------+------------------------+--------------------+----------------+
|NL99RABO9999999999 |7                       |63141.81            |SENDER          |
|NL11ABNA0111111111 |7                       |60657.43999999999   |RECEIVER        |
|NL44SNSB0444444444 |13                      |113251.24999999999  |RECEIVER        |
|NL33RABO0333333333 |14                      |118789.05000000002  |RECEIVER        |
|NL19ABNA1919191919 |22                      |179939.31           |RECEIVER        |
|NL55TRIO0555555555 |8                       |67327.33            |RECEIVER        |
|NL22INGB0222222222 |12                     

In [0]:
# print("\n3️⃣  Detecting circular transactions (rolling 30-day window)...")
# print("   Using rolling edges — cycles can span multiple days")

# from pyspark.sql.functions import broadcast
# from pyspark.sql.types import StructType, StructField, StringType
# import builtins

# # ── Build rolling undirected edges for component detection ─────
# rolling_directed = edges_rolling.select(
#     col("src").alias("id"),
#     col("dst").alias("neighbor")
# )

# rolling_reversed = edges_rolling.select(
#     col("dst").alias("id"),
#     col("src").alias("neighbor")
# )

# rolling_undirected = rolling_directed.unionByName(
#     rolling_reversed
# ).distinct()

# # ── Build rolling nodes ────────────────────────────────────────
# # Collect all unique IBANs seen in the rolling window
# rolling_nodes = rolling_undirected.select("id").distinct()

# # ── Run connected components on rolling graph ──────────────────
# # This gives us clusters based on 30 days of activity
# # not just today — so multi-day cycles appear in same cluster

# rolling_components = rolling_nodes.withColumn(
#     "component", col("id")
# )

# for i in range(8):   # more iterations for larger rolling graph
#     updated = rolling_components.alias("c").join(
#         rolling_undirected.alias("e"),
#         col("c.id") == col("e.id"),
#         "left"
#     ).join(
#         rolling_components.select(
#             col("id").alias("neighbor"),
#             col("component").alias("neighbor_component")
#         ),
#         "neighbor",
#         "left"
#     ).withColumn(
#         "new_component",
#         F.when(
#             col("neighbor_component").isNull(),
#             col("c.component")
#         ).otherwise(
#             F.least(col("c.component"), col("neighbor_component"))
#         )
#     ).groupBy("c.id").agg(
#         F.min("new_component").alias("component")
#     )

#     changes = updated.join(
#         rolling_components.select(
#             col("id"),
#             col("component").alias("old_component")
#         ),
#         "id"
#     ).filter(col("component") != col("old_component")).count()

#     rolling_components = updated

#     if changes == 0:
#         print(f"   Rolling components converged after {i+1} iterations ✅")
#         break

# # ── Cluster sizes for rolling graph ───────────────────────────
# rolling_cluster_sizes = rolling_components.groupBy("component").agg(
#     F.count("id").alias("cluster_size")
# ).withColumn(
#     # Cap hops at 8 for performance
#     # A 30-day cycle through 8 accounts is already extreme
#     "max_hops",
#     F.when(col("cluster_size") > 8, 8).otherwise(col("cluster_size"))
# )

# print("\n   Rolling cluster size distribution:")
# rolling_cluster_sizes.groupBy("max_hops").agg(
#     F.count("component").alias("num_clusters")
# ).orderBy("max_hops").show()

# # ── Path building on rolling edges ────────────────────────────
# # Same iterative path extension as before
# # but now using edges_rolling instead of today's edges only

# rolling_paths = edges_rolling.join(
#     rolling_components.select(
#         col("id").alias("src"),
#         col("component").alias("path_component")
#     ),
#     "src"
# ).select(
#     col("src").alias("start"),
#     col("dst").alias("current_end"),
#     lit(1).alias("path_length"),
#     col("path_component")
# )

# all_circular_accounts = spark.createDataFrame(
#     spark.sparkContext.emptyRDD(),
#     StructType([StructField("account_id", StringType(), True)])
# )

# cycle_summary = []

# max_global_hops = rolling_cluster_sizes.agg(
#     F.max("max_hops").alias("max_hops")
# ).collect()[0]["max_hops"]

# if max_global_hops is None:
#     max_global_hops = 4

# print(f"\n   Max hops: {max_global_hops}")

# for hop in range(2, max_global_hops + 1):

#     eligible_clusters = rolling_cluster_sizes.filter(
#         col("max_hops") >= hop
#     ).select(col("component").alias("path_component"))

#     eligible_paths = rolling_paths.join(
#         broadcast(eligible_clusters), "path_component"
#     ).filter(col("path_length") == hop - 1)

#     extended = eligible_paths.join(
#         edges_rolling.select(
#             col("src").alias("current_end"),
#             col("dst").alias("next_end")
#         ),
#         "current_end"
#     ).select(
#         col("start"),
#         col("next_end").alias("current_end"),
#         lit(hop).alias("path_length"),
#         col("path_component")
#     )

#     # Cycle = path returns to where it started
#     cycles_at_this_hop = extended.filter(
#         col("start") == col("current_end")
#     ).select(
#         col("start").alias("account_id")
#     ).distinct()

#     cycle_count = cycles_at_this_hop.count()
#     cycle_summary.append((hop, cycle_count))
#     print(f"   Hop {hop}: {cycle_count:,} cycle accounts found")

#     if cycle_count > 0:
#         all_circular_accounts = all_circular_accounts.unionByName(
#             cycles_at_this_hop
#         )

#     non_cycle_paths = extended.filter(
#         col("start") != col("current_end")
#     ).select(
#         "start", "current_end", "path_length", "path_component"
#     )

#     rolling_paths = rolling_paths.unionByName(non_cycle_paths)

# # ── Final result ───────────────────────────────────────────────
# circular_accounts = all_circular_accounts.distinct()
# total_circular    = circular_accounts.count()

# print(f"\n   ✅ Cycle detection summary:")
# for hop, cnt in cycle_summary:
#     print(f"      └── {hop}-hop cycles: {cnt:,} accounts")
# print(f"   ✅ Total unique circular accounts: {total_circular:,}")

# if total_circular > 0:
#     # Show which accounts are in cycles — useful for investigation
#     circular_accounts.join(
#         rolling_components.select(
#             col("id").alias("account_id"),
#             "component"
#         ),
#         "account_id"
#     ).show(10, truncate=False)

print("\n3️⃣  Detecting circular transactions (rolling 30-day window)...")

from pyspark.sql.types import StructType, StructField, StringType

# ── TEST MODE: cap data for fast execution ─────────────────────
edges_rolling = edges_rolling.select("src", "dst") \
    .distinct() \
    .limit(1000) \
    .cache()
edges_rolling.count()
print(f"✅ TEST MODE: capped to 1000 edges")

# ── Step 1: find accounts that both send AND receive ───────────
senders   = edges_rolling.select(col("src").alias("id")).distinct()
receivers = edges_rolling.select(col("dst").alias("id")).distinct()

bidirectional = senders.join(receivers, "id").cache()
total_bidirectional = bidirectional.count()
print(f"   Bidirectional accounts (cycle candidates): {total_bidirectional:,}")

# ── Step 2: build undirected edges ────────────────────────────
undirected = edges_rolling.select(
    col("src").alias("id"), col("dst").alias("neighbor")
).unionByName(
    edges_rolling.select(
        col("dst").alias("id"), col("src").alias("neighbor")
    )
).distinct().cache()
undirected.count()

# ── Step 3: connected components (3 iterations, no checkpoint) ─
components = undirected.select("id").distinct() \
    .withColumn("component", col("id"))

for i in range(3):
    updated = components.join(
        undirected.join(
            components.select(
                col("id").alias("neighbor"),
                col("component").alias("nc")
            ), "neighbor"
        ).groupBy("id").agg(F.min("nc").alias("min_nc")),
        "id", "left"
    ).withColumn(
        "component",
        F.least(col("component"), col("min_nc"))
    ).select("id", "component")

    # ── Break lineage by materialising through pandas ──────────
    # Cuts Spark's growing execution plan without needing
    # a checkpoint directory
    updated = spark.createDataFrame(updated.toPandas())

    changes = updated.join(
        components.select(
            col("id"), col("component").alias("old")
        ), "id"
    ).filter(col("component") != col("old")).count()

    components = updated
    print(f"   Iteration {i+1}: {changes:,} changes")
    if changes == 0:
        print(f"   ✅ Converged after {i+1} iterations")
        break

components = components.cache()
components.count()

# ── Step 4: find real cycles ───────────────────────────────────
# Cycle = component with 3+ nodes where at least one node
# is bidirectional (sends AND receives)
component_sizes = components.groupBy("component").agg(
    F.count("id").alias("size")
)

real_cycles = component_sizes.filter(col("size") >= 3) \
    .select("component")

circular_accounts = bidirectional.join(
    components, "id"
).join(
    real_cycles, "component"
).select(
    col("id").alias("account_id")
).distinct()

total_circular = circular_accounts.count()
print(f"\n   ✅ Total circular accounts: {total_circular:,}")

if total_circular > 0:
    circular_accounts.join(
        components.select(
            col("id").alias("account_id"),
            "component"
        ),
        "account_id"
    ).join(
        component_sizes.select(
            "component",
            col("size").alias("cycle_size")
        ),
        "component"
    ).orderBy(col("cycle_size").desc()) \
     .show(10, truncate=False)
else:
    print("   ℹ️  No circular accounts detected in test sample")
    print("   (expected if test data has < 3 nodes in any component)")

# ── Cleanup ────────────────────────────────────────────────────
edges_rolling.unpersist()
undirected.unpersist()
bidirectional.unpersist()
components.unpersist()



3️⃣  Detecting circular transactions (rolling 30-day window)...
✅ TEST MODE: capped to 1000 edges
   Bidirectional accounts (cycle candidates): 7
   Iteration 1: 945 changes
   Iteration 2: 152 changes
   Iteration 3: 2 changes

   ✅ Total circular accounts: 7
+----------------------------+------------------+----------+
|component                   |account_id        |cycle_size|
+----------------------------+------------------+----------+
|NL10RABO10101010101010101010|NL99RABO9999999999|23        |
|NL10ABNA1010101010          |NL80INGB8080808080|6         |
|NL10ABNA1010101010          |NL10ABNA1010101010|6         |
|NL10ABNA1010101010          |NL30RABO3030303030|6         |
|NL10ABNA1010101010          |NL20INGB2020202020|6         |
|NL55ABNA5555555555          |NL64BUNQ6464646464|4         |
|NL55ABNA5555555555          |NL73TRIO7373737373|4         |
+----------------------------+------------------+----------+



DataFrame[id: string, component: string]

In [0]:
# ── Network coherence for young accounts ──────────────────────
# The question is not "how much does this account move"
# The question is "does who they pay make economic sense"
#
# We look only ONE hop away — direct counterparties only.
# Cluster membership is NOT used because transitive connections
# (startup → AWS → 10,000 other companies) make clusters
# meaningless for this purpose.
#
# Three signals:
# 1. Direct counterparty risk ratio — what fraction of accounts
#    this company DIRECTLY pays are already flagged as criminal
# 2. Counterparty sanctions density — what fraction of direct
#    counterparties are on a sanctions list
# 3. Purpose code coherence — does the payment variety look
#    like a real operating business

print("📊 Computing network coherence for young accounts...")

# ── Build suspicious account lookup from already-detected patterns
# These are all the accounts your system has already flagged
# in the pattern detection cell above (Cell 8)
df_suspicious_accounts = structuring_flagged.select(
    F.col("structuring_account").alias("account_id")
).unionByName(
    chain_nodes_rolling.select(F.col("id").alias("account_id"))
).unionByName(
    circular_accounts.select("account_id")
).unionByName(
    nodes.filter(F.col("sanctions_hit") == True)
         .select(F.col("id").alias("account_id"))
).distinct() \
 .withColumn("is_suspicious", F.lit(True))

# ── Identify young accounts — active less than 90 days
young_accounts = df_account_stats.filter(
    F.col("account_age_days") < 90
).select("sender_iban", "account_age_days")

young_count = young_accounts.count()
print(f"   Young accounts found (< 90 days): {young_count:,}")

if young_count == 0:
    print("⚠️  No young accounts — skipping coherence check")
    # Create empty DataFrame with correct schema so joins below don't fail
    from pyspark.sql.types import (
        StructType, StructField, StringType, DoubleType,
        IntegerType, BooleanType
    )
    df_coherence_risk = spark.createDataFrame([], StructType([
        StructField("sender_iban",                    StringType(),  True),
        StructField("account_age_days",               DoubleType(),  True),
        StructField("total_direct_counterparties",    IntegerType(), True),
        StructField("suspicious_direct_counterparties", IntegerType(), True),
        StructField("direct_counterparty_risk_ratio", DoubleType(),  True),
        StructField("counterparty_sanctions_density", DoubleType(),  True),
        StructField("sanctioned_counterparty_count",  IntegerType(), True),
        StructField("purpose_code_variety",           IntegerType(), True),
        StructField("has_operational_payments",       BooleanType(), True),
        StructField("coherence_risk_score",           IntegerType(), True),
        StructField("coherence_risk_flag",            BooleanType(), True),
    ]))
else:
    # ── Signal 1: Direct counterparty risk ratio ───────────────
    # For each young sender, check their direct counterparties
    # against the suspicious accounts list — one hop only

    young_sender_edges = edges.join(
        young_accounts.select(F.col("sender_iban").alias("src")),
        "src"
    ).select("src", "dst")

    direct_counterparty_risk = young_sender_edges.join(
        df_suspicious_accounts,
        young_sender_edges["dst"] == df_suspicious_accounts["account_id"],
        "left"
    ).groupBy("src").agg(
        F.count("dst").alias("total_direct_counterparties"),
        F.sum(
            F.when(F.col("is_suspicious") == True, 1).otherwise(0)
        ).alias("suspicious_direct_counterparties")
    ).withColumn(
        # 0.0 = all counterparties are clean (legitimate startup)
        # 0.75 = 6 of 8 counterparties are flagged criminals
        "direct_counterparty_risk_ratio",
        F.when(
            F.col("total_direct_counterparties") > 0,
            F.col("suspicious_direct_counterparties").cast("double") /
            F.col("total_direct_counterparties").cast("double")
        ).otherwise(0.0)
    ).select(
        F.col("src").alias("sender_iban"),
        "total_direct_counterparties",
        "suspicious_direct_counterparties",
        "direct_counterparty_risk_ratio"
    )

    # ── Signal 2: Counterparty sanctions density ───────────────
    # What fraction of direct counterparties are sanctions-hit?
    # Uses your nodes DataFrame which already has sanctions_hit

    counterparty_sanctions = young_sender_edges.join(
        nodes.select(
            F.col("id").alias("dst"),
            F.col("sanctions_hit").alias("counterparty_sanctioned")
        ),
        "dst"
    ).groupBy("src").agg(
        F.count("dst").alias("total_cp_s"),
        F.sum(
            F.when(F.col("counterparty_sanctioned"), 1).otherwise(0)
        ).alias("sanctioned_counterparty_count")
    ).withColumn(
        "counterparty_sanctions_density",
        F.when(
            F.col("total_cp_s") > 0,
            F.col("sanctioned_counterparty_count").cast("double") /
            F.col("total_cp_s").cast("double")
        ).otherwise(0.0)
    ).select(
        F.col("src").alias("sender_iban"),
        "counterparty_sanctions_density",
        "sanctioned_counterparty_count"
    )

    # ── Signal 3: Purpose code coherence ──────────────────────
    # Real businesses have diverse payment types.
    # SALA = salary, RENT = rent, SUPP = supplier, UTIL = utilities
    # These are unavoidable operational payments for real businesses.
    # A company with none of these has no real operational footprint.

    purpose_variety = edges.join(
        young_accounts.select(F.col("sender_iban").alias("src")),
        "src"
    ).groupBy("src").agg(
        F.countDistinct("purpose_code").alias("purpose_code_variety"),
        F.collect_set("purpose_code").alias("purpose_codes_used")
    ).withColumn(
        "has_operational_payments",
        F.arrays_overlap(
            F.col("purpose_codes_used"),
            F.array(
                F.lit("SALA"), F.lit("RENT"),
                F.lit("UTIL"), F.lit("SUPP")
            )
        )
    ).select(
        F.col("src").alias("sender_iban"),
        "purpose_code_variety",
        "has_operational_payments"
    )

    # ── Combine all three signals ──────────────────────────────
    df_coherence_risk = young_accounts.join(
        direct_counterparty_risk, "sender_iban", "left"
    ).join(
        counterparty_sanctions, "sender_iban", "left"
    ).join(
        purpose_variety, "sender_iban", "left"
    ).fillna(0.0, subset=[
        "direct_counterparty_risk_ratio",
        "counterparty_sanctions_density"
    ]).fillna(0, subset=[
        "purpose_code_variety",
        "total_direct_counterparties",
        "suspicious_direct_counterparties",
        "sanctioned_counterparty_count"
    ]).fillna(False, subset=["has_operational_payments"]) \
    .withColumn(
        "coherence_risk_score",
        # Pays known criminals directly — strongest signal
        (F.when(
            F.col("direct_counterparty_risk_ratio") > 0.3, 4
        ).otherwise(0)) +
        # Any sanctioned direct counterparties
        (F.when(
            F.col("counterparty_sanctions_density") > 0.1, 4
        ).otherwise(0)) +
        # Only 1 type of payment — not behaving like real business
        (F.when(
            F.col("purpose_code_variety") <= 1, 2
        ).otherwise(0)) +
        # No operational payments at all — no salary, rent, suppliers
        (F.when(
            F.col("has_operational_payments") == False, 2
        ).otherwise(0))
    ).withColumn(
        # Score >= 5 requires at least two signals firing together
        # Startup with low purpose variety but clean counterparties = 2, not flagged
        # Criminal with flagged counterparties + no operations = 8, flagged
        "coherence_risk_flag",
        F.col("coherence_risk_score") >= 5
    )

    flagged = df_coherence_risk.filter(F.col("coherence_risk_flag")).count()
    print(f"✅ Young accounts analysed:    {young_count:,}")
    print(f"🚨 Coherence risk flagged:     {flagged:,}")
    df_coherence_risk.filter(F.col("coherence_risk_flag")) \
        .select(
            "sender_iban",
            "account_age_days",
            "direct_counterparty_risk_ratio",
            "counterparty_sanctions_density",
            "purpose_code_variety",
            "has_operational_payments",
            "coherence_risk_score"
        ).show(5, truncate=False)

📊 Computing network coherence for young accounts...
   Young accounts found (< 90 days): 1,282
✅ Young accounts analysed:    1,282
🚨 Coherence risk flagged:     110
+------------------+-------------------+------------------------------+------------------------------+--------------------+------------------------+--------------------+
|sender_iban       |account_age_days   |direct_counterparty_risk_ratio|counterparty_sanctions_density|purpose_code_variety|has_operational_payments|coherence_risk_score|
+------------------+-------------------+------------------------------+------------------------------+--------------------+------------------------+--------------------+
|NL20INGB2020202020|9.080555555555556  |1.0                           |0.0                           |1                   |false                   |8                   |
|NL91RABO9191919191|0.24027777777777778|1.0                           |0.0                           |1                   |false                   |8      

In [0]:
print("📊 Computing cluster metrics using DETECTED patterns...")

# Join structuring detection to clusters
structuring_per_cluster = structuring_flagged.join(
    components.select(
        col("id").alias("structuring_account"),
        "component"
    ),
    "structuring_account"
).groupBy("component").agg(
    F.count("structuring_account").alias("structuring_accounts_count"),
    F.sum("near_threshold_total").alias("structuring_total_amount")
)

# Layering already grouped by component from detection above
layering_per_cluster = layering_by_cluster_confirmed

# Circular accounts joined to clusters
# Use rolling_components for circular join —
# the cycle was detected in the rolling graph not today's graph.
# Then map back to today's component via shared IBAN membership.

circular_per_cluster = circular_accounts.join(
    rolling_components_l.select(
        col("id").alias("account_id"),
        col("component").alias("rolling_component")
    ),
    "account_id"
).join(
    # Map rolling component → today's component
    # via accounts that appear in both graphs
    components.select(
        col("id").alias("account_id"),
        "component"
    ),
    "account_id",
    "left"
).groupBy("component").agg(
    F.countDistinct("account_id").alias("circular_accounts_count")
).filter(
    col("component").isNotNull()
)

# ── Base cluster metrics (volume, value) ───────────────────────
edges_with_components = edges.join(
    components.select(
        col("id").alias("src"),
        col("component").alias("src_component")
    ),
    "src"
)

cluster_metrics = edges_with_components.groupBy("src_component").agg(
    F.count("transaction_id").alias("transaction_count"),
    F.sum("amount_eur").alias("total_amount_eur"),
    F.avg("amount_eur").alias("avg_amount_eur"),
    F.max("amount_eur").alias("max_amount_eur"),
    F.sum(F.when(col("sanctions_hit") == True, 1).otherwise(0))
        .alias("sanctions_hit_count"),
    F.sum(F.when(col("young_account_high_value") == True, 1).otherwise(0))
        .alias("high_value_count"),
    F.countDistinct("src").alias("unique_senders"),
    F.countDistinct("dst").alias("unique_receivers"),
)

# ── Join detected pattern counts (not labels!) ─────────────────
cluster_metrics = cluster_metrics \
    .join(
        structuring_per_cluster.withColumnRenamed("component", "src_component"),
        "src_component", "left"
    ) \
    .join(
        layering_per_cluster.withColumnRenamed("component", "src_component"),
        "src_component", "left"
    ) \
    .join(
        circular_per_cluster.withColumnRenamed("component", "src_component"),
        "src_component", "left"
    ) \
    .fillna(0, subset=[
        "structuring_accounts_count",
        "structuring_total_amount",
        "layering_chain_length",
        "circular_accounts_count"
    ])

print(f"✅ Cluster metrics computed: {cluster_metrics.count():,} clusters")
cluster_metrics.show(5)

📊 Computing cluster metrics using DETECTED patterns...
✅ Cluster metrics computed: 842 clusters
+------------------+-----------------+----------------+--------------+--------------+-------------------+----------------+--------------+----------------+--------------------------+------------------------+---------------------+------------------------------+------------------+-------------------+-----------------------+
|     src_component|transaction_count|total_amount_eur|avg_amount_eur|max_amount_eur|sanctions_hit_count|high_value_count|unique_senders|unique_receivers|structuring_accounts_count|structuring_total_amount|layering_chain_length|suspicious_purpose_chain_nodes|layering_confirmed|spans_multiple_days|circular_accounts_count|
+------------------+-----------------+----------------+--------------+--------------+-------------------+----------------+--------------+----------------+--------------------------+------------------------+---------------------+------------------------------

In [0]:
# ── Aggregate statistical anomaly flags to cluster level ───────
# Joins onto cluster_metrics before risk scoring runs

stat_per_cluster = edges.join(
    components.select(
        col("id").alias("src"),
        col("component").alias("src_component")
    ),
    "src"
).groupBy("src_component").agg(
    F.sum(
        F.when(F.col("statistical_anomaly") == True, 1).otherwise(0)
    ).alias("statistical_anomaly_count"),
    F.sum(
        F.when(F.col("young_account_high_value") == True, 1).otherwise(0)
    ).alias("young_account_high_value_count"),
)

# Join onto cluster_metrics
cluster_metrics = cluster_metrics.join(
    stat_per_cluster, "src_component", "left"
).fillna(0, subset=[
    "statistical_anomaly_count",
    "young_account_high_value_count"
])

print("✅ Statistical anomaly counts joined to cluster metrics")
cluster_metrics.select(
    "src_component",
    "statistical_anomaly_count",
    "young_account_high_value_count"
).filter(F.col("statistical_anomaly_count") > 0).show(5)

✅ Statistical anomaly counts joined to cluster metrics
+------------------+-------------------------+------------------------------+
|     src_component|statistical_anomaly_count|young_account_high_value_count|
+------------------+-------------------------+------------------------------+
|NL10ABNA6953797985|                        1|                             1|
|NL10RABO9150634714|                        2|                             3|
+------------------+-------------------------+------------------------------+



In [0]:
print("📈 Computing AML risk scores...")

from pyspark.sql.functions import least, when, col, lit, current_timestamp

scored_clusters = cluster_metrics.withColumn(
    "risk_score",
    (col("structuring_accounts_count")    * 15) +
    (col("layering_chain_length")         * 20) +
    (col("circular_accounts_count")       * 20) +
    (F.when(col("sanctions_hit_count") >= 2, 30)
    .when(col("sanctions_hit_count") == 1, 10)
    .otherwise(0)) +
    (col("high_value_count")              * 5)  +
    (col("statistical_anomaly_count")     * 10) 
).withColumn(
    "risk_score",
    least(col("risk_score"), lit(100))
).withColumn(
    "risk_category",
    when(col("risk_score") >= 70, "HIGH")
    .when(col("risk_score") >= 40, "MEDIUM")
    .otherwise("LOW")
).withColumn(
    "primary_typology",
    when(col("sanctions_hit_count") > 0,        "SANCTIONS_EXPOSURE")
    .when(col("circular_accounts_count") > 0,    "CIRCULAR_TRADING_DETECTED")
    .when(col("layering_chain_length") > 0,      "LAYERING_DETECTED")
    .when(col("structuring_accounts_count") > 0, "STRUCTURING_DETECTED")
    .when(
        col("statistical_anomaly_count") > 0,
        "STATISTICAL_ANOMALY_DETECTED"
    )
    .otherwise("NORMAL")
).withColumn(
    "scored_at",
    F.current_timestamp()
)

print("✅ Risk scores computed")
scored_clusters.groupBy("risk_category").count().show()
scored_clusters.select(
    "src_component", "transaction_count", "risk_score", 
    "risk_category", "primary_typology"
).orderBy(col("risk_score").desc()).show(10)

📈 Computing AML risk scores...
✅ Risk scores computed
+-------------+-----+
|risk_category|count|
+-------------+-----+
|          LOW|  836|
|       MEDIUM|    1|
|         HIGH|    5|
+-------------+-----+

+--------------------+-----------------+----------+-------------+--------------------+
|       src_component|transaction_count|risk_score|risk_category|    primary_typology|
+--------------------+-----------------+----------+-------------+--------------------+
|NL10RABO101010101...|               76|       100|         HIGH|CIRCULAR_TRADING_...|
|  NL50TRIO5050505050|               22|       100|         HIGH|              NORMAL|
|  NL10ABNA1010101010|               64|       100|         HIGH|CIRCULAR_TRADING_...|
|  NL55ABNA5555555555|               40|       100|         HIGH|CIRCULAR_TRADING_...|
|  NL46INGB4646464646|               16|        80|         HIGH|              NORMAL|
|  NL10RABO9014207737|               13|        65|       MEDIUM|              NORMAL|
|  NL10R

In [0]:
print("🚨 Generating alerts for high risk clusters...")

# Only alert on HIGH and MEDIUM risk clusters
alerts = scored_clusters.filter(
    col("risk_category").isin(["HIGH", "MEDIUM"])
).withColumn(
    "alert_id",
    F.concat(
        lit("CLARITY-ALERT-"),
        F.date_format(F.current_timestamp(), "yyyyMMdd"),
        lit("-"),
        col("src_component").cast("string")
    )
).withColumn(
    "alert_status",
    lit("OPEN")
).withColumn(
    "assigned_to",
    lit("AML_INVESTIGATION_TEAM")
).withColumn(
    "alert_narrative",
    F.concat(
        lit("Suspicious cluster detected. Typology: "),
        col("primary_typology"),
        lit(". Transactions: "),
        col("transaction_count").cast("string"),
        lit(". Total amount: EUR "),
        F.round(col("total_amount_eur"), 2).cast("string"),
        lit(". Risk score: "),
        col("risk_score").cast("string"),
        lit("/100")
    )
)

alert_count = alerts.count()
print(f"✅ Alerts generated: {alert_count:,}")

print("\n🚨 Sample alerts:")
alerts.select(
    "alert_id",
    "risk_category",
    "risk_score",
    "primary_typology",
    "transaction_count",
    "total_amount_eur",
    "alert_narrative"
).show(5, truncate=False)

🚨 Generating alerts for high risk clusters...
✅ Alerts generated: 6

🚨 Sample alerts:
+---------------------------------------------------+-------------+----------+-------------------------+-----------------+------------------+-------------------------------------------------------------------------------------------------------------------------------------+
|alert_id                                           |risk_category|risk_score|primary_typology         |transaction_count|total_amount_eur  |alert_narrative                                                                                                                      |
+---------------------------------------------------+-------------+----------+-------------------------+-----------------+------------------+-------------------------------------------------------------------------------------------------------------------------------------+
|CLARITY-ALERT-20260703-NL10RABO10101010101010101010|HIGH         |100       |CIRCULAR

In [0]:
print("🏦 Flagging individual accounts...")

# ── Account-level structuring flag ────────────────────────────
structuring_account_flags = structuring_flagged.select(
    col("structuring_account").alias("account_id"),
    lit(True).alias("is_structuring_account"),
    col("near_threshold_txn_count").alias("structuring_txn_count"),
    col("near_threshold_total").alias("structuring_total_amount"),
    col("structuring_role")   # SENDER or RECEIVER
)

# ── Account-level layering flag ───────────────────────────────
layering_account_flags = chain_nodes_rolling.select(
    col("id").alias("account_id"),
    lit(True).alias("is_layering_account"),
    col("degree").alias("connection_degree")
).join(
    components.select(
        col("id").alias("account_id"),
        "component"
    ),
    "account_id"
).join(
    layering_by_cluster.select(
        col("component"),
        col("layering_chain_length")
    ),
    "component",
    "left"
)

# ── Account-level circular flag ───────────────────────────────
circular_account_flags = circular_accounts.select(
    col("account_id"),
    lit(True).alias("is_circular_account")
)

# ── Build master account risk table ───────────────────────────
# Start with all nodes and their cluster scores
# Build nodes_with_components BEFORE account risk computation

nodes_with_components = nodes.join(
    components.select("id", "component"),
    "id"
).join(
    scored_clusters.select(
        col("src_component").alias("component"),
        "risk_score",
        "risk_category",
        "primary_typology"
    ),
    "component",
    "left"
)


account_risk = nodes_with_components.select(
    col("id").alias("account_id"),
    col("name").alias("account_name"),
    col("bic").alias("account_bic"),
    col("sanctions_hit"),
    col("kvk_registered"),
    col("kvk_status"),
    col("component"),
    col("risk_score").alias("cluster_risk_score"),
    col("risk_category").alias("cluster_risk_category"),
    col("primary_typology").alias("cluster_primary_typology")
)

# Join individual pattern flags
account_risk = account_risk \
    .join(
        structuring_account_flags,
        "account_id", "left"
    ) \
    .join(
        layering_account_flags.select(
            "account_id",
            "is_layering_account",
            "connection_degree",
            "layering_chain_length"
        ),
        "account_id", "left"
    ) \
    .join(
        circular_account_flags,
        "account_id", "left"
    ) \
    .fillna(False, subset=[
        "is_structuring_account",
        "is_layering_account",
        "is_circular_account"
    ]) \
    .fillna(0, subset=[
        "structuring_txn_count",
        "structuring_total_amount",
        "connection_degree",
        "layering_chain_length"
    ])

# ── Add statistical flags to account risk ─────────────────────
# Add this block AFTER the existing .fillna() calls
# and BEFORE the account_risk_score withColumn chain

# ── Join statistical anomaly flags ─────────────────────────────
df_stat_flags = transactions.groupBy(
    F.col("sender_iban").alias("account_id")
).agg(
    F.max("statistical_anomaly").alias("has_statistical_anomaly"),
    F.max("amount_zscore").alias("max_zscore"),
    F.first("hist_mean_amount").alias("hist_mean_amount"),
    F.first("account_age_days").alias("account_age_days")
)

account_risk = account_risk.join(
    df_stat_flags, "account_id", "left"
).fillna(False, subset=["has_statistical_anomaly"]) \
 .fillna(0.0,   subset=["max_zscore", "hist_mean_amount", "account_age_days"])

# ── Join network coherence flags ───────────────────────────────
account_risk = account_risk.join(
    df_coherence_risk.select(
        F.col("sender_iban").alias("account_id"),
        "coherence_risk_flag",
        "coherence_risk_score",
        "direct_counterparty_risk_ratio",
        "counterparty_sanctions_density",
        "purpose_code_variety",
        "has_operational_payments"
    ),
    "account_id",
    "left"
).fillna(False, subset=[
    "coherence_risk_flag",
    "has_operational_payments"
]).fillna(0.0, subset=[
    "direct_counterparty_risk_ratio",
    "counterparty_sanctions_density"
]).fillna(0, subset=[
    "coherence_risk_score",
    "purpose_code_variety"
])
# ── Compute individual account risk score ──────────────────────
# Separate from cluster score — this is THIS account's own risk
account_risk = account_risk.withColumn(
    "account_risk_score",
    (when(col("sanctions_hit"),             lit(50)).otherwise(lit(0))) +
    (when(col("is_structuring_account"),    lit(25)).otherwise(lit(0))) +
    (when(col("is_layering_account"),       lit(20)).otherwise(lit(0))) +
    (when(col("is_circular_account"),       lit(20)).otherwise(lit(0))) +
    (when(~col("kvk_registered"),              lit(15)).otherwise(lit(0))) +
    (when(col("has_statistical_anomaly"),       lit(25)).otherwise(lit(0))) +  # ← ADD
    (when(col("coherence_risk_flag"),           lit(35)).otherwise(lit(0)))    # ← ADD
).withColumn(
    "account_risk_score",
    least(col("account_risk_score"), lit(100))
).withColumn(
    "account_risk_category",
    when(col("account_risk_score") >= 70, "HIGH")
    .when(col("account_risk_score") >= 40, "MEDIUM")
    .otherwise("LOW")
).withColumn(
    "flagged_for_investigation",
    col("account_risk_score") >= 40
).withColumn(
    "account_flags_summary",
    F.concat_ws(", ",
        when(col("sanctions_hit"),          lit("SANCTIONS_HIT")),
        when(col("is_structuring_account"), lit("STRUCTURING")),
        when(col("is_layering_account"),    lit("LAYERING")),
        when(col("is_circular_account"),    lit("CIRCULAR")),
        when(~col("kvk_registered"),        lit("UNREGISTERED_ENTITY")),
        when(col("has_statistical_anomaly"), lit("STATISTICAL_ANOMALY")),
        when(col("coherence_risk_flag"),     lit("COHERENCE_RISK")),   # ← ADD
    )
).withColumn(
    "processed_year",  lit(YEAR)
).withColumn(
    "processed_month", lit(MONTH)
).withColumn(
    "processed_day",   lit(DAY)
)

total_accounts       = account_risk.count()
high_risk_accounts   = account_risk.filter(col("account_risk_category") == "HIGH").count()
medium_risk_accounts = account_risk.filter(col("account_risk_category") == "MEDIUM").count()
flagged_accounts     = account_risk.filter(col("flagged_for_investigation")).count()

print(f"✅ Account risk table built")
print(f"   Total accounts:        {total_accounts:,}")
print(f"   High risk accounts:    {high_risk_accounts:,}")
print(f"   Medium risk accounts:  {medium_risk_accounts:,}")
print(f"   Flagged for review:    {flagged_accounts:,}")

print("\n🔍 Sample high risk accounts:")
account_risk.filter(col("account_risk_category") == "HIGH") \
    .select(
        "account_id",
        "account_name",
        "account_risk_score",
        "account_risk_category",
        "account_flags_summary",
        "cluster_risk_score"
    ).show(10, truncate=False)

🏦 Flagging individual accounts...
✅ Account risk table built
   Total accounts:        1,842
   High risk accounts:    14
   Medium risk accounts:  171
   Flagged for review:    185

🔍 Sample high risk accounts:
+------------------+------------------+------------------+---------------------+-------------------------------------------------------+------------------+
|account_id        |account_name      |account_risk_score|account_risk_category|account_flags_summary                                  |cluster_risk_score|
+------------------+------------------+------------------+---------------------+-------------------------------------------------------+------------------+
|NL80INGB8080808080|LAYERING NODE H BV|90                |HIGH                 |LAYERING, CIRCULAR, UNREGISTERED_ENTITY, COHERENCE_RISK|100               |
|NL20INGB2020202020|LAYERING NODE B BV|90                |HIGH                 |LAYERING, CIRCULAR, UNREGISTERED_ENTITY, COHERENCE_RISK|100               |
|NL10ABN

In [0]:
print(account_risk.columns)

['account_id', 'account_name', 'account_bic', 'sanctions_hit', 'kvk_registered', 'kvk_status', 'component', 'cluster_risk_score', 'cluster_risk_category', 'cluster_primary_typology', 'is_structuring_account', 'structuring_txn_count', 'structuring_total_amount', 'structuring_role', 'is_layering_account', 'connection_degree', 'layering_chain_length', 'is_circular_account', 'has_statistical_anomaly', 'max_zscore', 'hist_mean_amount', 'account_age_days', 'coherence_risk_flag', 'coherence_risk_score', 'direct_counterparty_risk_ratio', 'counterparty_sanctions_density', 'purpose_code_variety', 'has_operational_payments', 'account_risk_score', 'account_risk_category', 'flagged_for_investigation', 'account_flags_summary', 'processed_year', 'processed_month', 'processed_day']


In [0]:
def write_snowflake(df, table_name, mode="overwrite", schema="GOLD"):
    """
    Write a Spark DataFrame to Snowflake.
    
    Args:
        df:         Spark DataFrame to write
        table_name: Target table name (e.g. "AML_ALERTS")
        mode:       "overwrite" (default) or "append"
        schema:     "GOLD" (default) or "ACTIONS"
    """
    df.write \
      .format("snowflake") \
      .options(**SNOWFLAKE_OPTIONS) \
      .option("sfSchema", schema) \
      .option("dbtable", table_name) \
      .mode(mode) \
      .save()
    
    count = df.count()
    print(f"✅ Written to Snowflake: CLARITY_AML.{schema}.{table_name} ({count:,} rows)")

In [0]:
import pandas as pd
spark.conf.set("spark.sql.shuffle.partitions", "8")

# ── Pre-cache heavy DataFrames before Snowflake writes ─────────
print("📦 Caching DataFrames...")
account_risk = account_risk.cache()
account_risk.count()
print(f"   ✅ account_risk cached")

transactions = transactions.cache()
transactions.count()
print(f"   ✅ transactions cached")

📦 Caching DataFrames...
   ✅ account_risk cached
   ✅ transactions cached


In [0]:
print("❄️  Writing Gold output to Snowflake...")
from pyspark.sql.functions import current_date

# ── Cache account_risk first ───────────────────────────────────
# Prevents Spark from recomputing the entire detection pipeline
# for every single write call
account_risk_pd = account_risk.toPandas()
print(f"   account_risk cached: {len(account_risk_pd):,} rows")

# ── 1. AML Alerts ──────────────────────────────────────────────
aml_alerts_pd = account_risk_pd[account_risk_pd["account_risk_score"] > 0][[
    "account_id", "account_risk_score", "account_risk_category",
    "account_flags_summary", "is_structuring_account", "is_layering_account",
    "is_circular_account", "coherence_risk_flag", "sanctions_hit",
    "has_statistical_anomaly", "flagged_for_investigation",
    "cluster_primary_typology"
]].rename(columns={
    "account_risk_score":     "risk_score",
    "account_risk_category":  "risk_tier",
    "is_structuring_account": "is_structuring",
    "is_layering_account":    "is_layering",
    "is_circular_account":    "is_circular",
    "has_statistical_anomaly":"statistical_anomaly",
    "flagged_for_investigation": "young_account_high_value",
    "cluster_primary_typology":  "alert_narrative",
    "coherence_risk_flag":    "is_fan_in"
})
aml_alerts_pd["alert_date"] = pd.Timestamp.now().date()

aml_alerts_sf = spark.createDataFrame(aml_alerts_pd)
write_snowflake(aml_alerts_sf, "AML_ALERTS")

# ── 2. Account risk scores ─────────────────────────────────────
scores_cols = [
    "account_id", "hist_mean_amount", "max_zscore", "layering_chain_length",
    "sanctions_hit", "kvk_registered", "account_risk_score",
    "account_risk_category", "cluster_risk_score", "cluster_risk_category",
    "cluster_primary_typology", "structuring_txn_count", "structuring_total_amount",
    "connection_degree", "is_circular_account", "has_statistical_anomaly",
    "coherence_risk_score", "account_age_days", "direct_counterparty_risk_ratio",
    "counterparty_sanctions_density", "purpose_code_variety",
    "has_operational_payments", "flagged_for_investigation"
]
scores_pd = account_risk_pd[scores_cols].rename(columns={
    "hist_mean_amount":    "hist_avg_amount",
    "max_zscore":          "z_score",
    "account_risk_score":  "risk_score",
    "account_risk_category": "risk_tier"
})
scores_pd["process_date"] = pd.Timestamp.now().date()

account_scores_sf = spark.createDataFrame(scores_pd)
write_snowflake(account_scores_sf, "ACCOUNT_RISK_SCORES")

# ── 3. Flagged transactions ────────────────────────────────────
# This was the one crashing — limit to flagged accounts only
# before joining so the dataset is small
flagged_account_ids = set(aml_alerts_pd["account_id"].tolist())

flagged_txns_sf = transactions.filter(
    col("sender_iban").isin(flagged_account_ids)
).select(
    col("transaction_id"),
    col("sender_iban"),
    col("sender_name"),
    col("receiver_iban"),
    col("receiver_name"),
    col("amount_eur"),
    col("purpose_code"),
    col("value_date").cast("date"),
    col("aml_pattern")
).limit(50000)   # hard cap — prevents driver OOM

# Add risk score via pandas join (cheaper than Spark join)
risk_map = aml_alerts_pd[["account_id", "risk_score", "account_flags_summary"]] \
    .rename(columns={"account_id": "sender_iban", "account_flags_summary": "flags"})

flagged_txns_pd = flagged_txns_sf.toPandas()
flagged_txns_pd = flagged_txns_pd.merge(risk_map, on="sender_iban", how="left")

write_snowflake(spark.createDataFrame(flagged_txns_pd), "FLAGGED_TRANSACTIONS")

# ── 4. Circular clusters ───────────────────────────────────────
if total_circular > 0:
    circular_pd = circular_accounts.join(
        components.select(col("id").alias("account_id"), "component"),
        "account_id"
    ).toPandas()
    circular_pd["detected_date"] = pd.Timestamp.now().date()
    write_snowflake(spark.createDataFrame(circular_pd), "CIRCULAR_CLUSTERS")
    print(f"   ✅ CIRCULAR_CLUSTERS: {len(circular_pd):,} accounts")
else:
    print("   ℹ️  No circular clusters")

# ── 5. Layering chains ─────────────────────────────────────────
layering_sf = chain_nodes_rolling.join(
    rolling_components_l.select("id", "component"), "id"
).join(
    layering_by_cluster.select(
        "component",
        col("layering_chain_length").alias("chain_length"),
        col("layering_confirmed").alias("confirmed")
    ), "component"
).select(
    col("id").alias("account_id"),
    col("component"),
    col("chain_length"),
    col("confirmed"),
    current_date().alias("detected_date")
).distinct()

layering_pd = layering_sf.toPandas()
write_snowflake(spark.createDataFrame(layering_pd), "LAYERING_CHAINS")

print("\n✅ All Gold tables written to Snowflake successfully")
print(f"   → AML_ALERTS:          {len(aml_alerts_pd):,} rows")
print(f"   → ACCOUNT_RISK_SCORES: {len(scores_pd):,} rows")
print(f"   → FLAGGED_TRANSACTIONS:{len(flagged_txns_pd):,} rows")
print(f"   → LAYERING_CHAINS:     {len(layering_pd):,} rows")

❄️  Writing Gold output to Snowflake...
   account_risk cached: 1,842 rows
✅ Written to Snowflake: CLARITY_AML.GOLD.AML_ALERTS (1,443 rows)
✅ Written to Snowflake: CLARITY_AML.GOLD.ACCOUNT_RISK_SCORES (1,842 rows)
✅ Written to Snowflake: CLARITY_AML.GOLD.FLAGGED_TRANSACTIONS (812 rows)
✅ Written to Snowflake: CLARITY_AML.GOLD.CIRCULAR_CLUSTERS (7 rows)
   ✅ CIRCULAR_CLUSTERS: 7 accounts
✅ Written to Snowflake: CLARITY_AML.GOLD.LAYERING_CHAINS (14 rows)

✅ All Gold tables written to Snowflake successfully
   → AML_ALERTS:          1,443 rows
   → ACCOUNT_RISK_SCORES: 1,842 rows
   → FLAGGED_TRANSACTIONS:812 rows
   → LAYERING_CHAINS:     14 rows
